In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import pywt
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks
from scipy.fft import fft


import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('before.csv',names=['time','x','y','z'],header=0)

count=0
for n in xs:
    count+=1
    plt.plot(n)
    plt.ylim(df['x'].min(),df['x'].max())
    plt.savefig(f'x\\{count}.png')
    plt.clf()

In [ ]:
def make_df(n1, column, sampling_frequency=10000):
    def slice_signal(signal, slice_length=1000):
        slices = [signal.iloc[i:i + slice_length] for i in range(0, len(signal), slice_length)]
        return slices[:-1]

    def extract_signal_features(signal, sampling_frequency=10000):
        signal = np.squeeze(np.asarray(signal))

        # Time-domain features
        features = {
            't_mean': round(np.mean(signal), 6),
            't_std': round(np.std(signal), 6),
            't_rms': round(np.sqrt(np.mean(signal**2)), 6),
            't_skew': round(skew(signal), 6),
            't_kurt': round(kurtosis(signal), 6),
            't_ptp': round(np.ptp(signal), 6),
            't_median': round(np.median(signal), 6),
            't_iqr': round(np.percentile(signal, 75) - np.percentile(signal, 25), 6),
            't_zero_crossings': (signal[:-1] * signal[1:] < 0).sum(),
        }

        # Frequency-domain features using FFT
        fft_values = fft(signal)
        frequencies = np.fft.fftfreq(len(fft_values), 1 / sampling_frequency)
        fft_magnitude = np.abs(fft_values)

        features['f_total_power'] = round(np.sum(fft_magnitude**2), 6)
        peak_frequency_idx = np.argmax(fft_magnitude[:len(fft_magnitude)//2])
        features['f_peak_freq'] = round(frequencies[peak_frequency_idx], 6)
        features['f_spectral_centroid'] = round(np.sum(frequencies * fft_magnitude) / (np.sum(fft_magnitude) + 1e-12), 6)
        features['f_spectral_bandwidth'] = round(np.sqrt(np.sum(((frequencies - features['f_spectral_centroid']) ** 2) * fft_magnitude) / (np.sum(fft_magnitude) + 1e-12)), 6)
        
        # Spectral Entropy
        normalized_magnitude = fft_magnitude / (np.sum(fft_magnitude) + 1e-12)
        features['f_spectral_entropy'] = round(-np.sum(normalized_magnitude * np.log2(normalized_magnitude + 1e-12)), 6)

        # Time-Frequency domain features (Simple approximation using FFT magnitudes)
        features['f_tf_energy_mean'] = round(np.mean(fft_magnitude), 6)
        features['f_tf_energy_std'] = round(np.std(fft_magnitude), 6)
        features['f_max_energy_freq'] = round(frequencies[np.argmax(fft_magnitude)], 6)

        # Power in Low, Mid, and High Frequency Bands
        low_band = (frequencies >= 0) & (frequencies < 100)
        mid_band = (frequencies >= 100) & (frequencies < 1000)
        high_band = (frequencies >= 1000)

        features['f_low_band_power'] = round(np.sum(fft_magnitude[low_band]**2), 6)
        features['f_mid_band_power'] = round(np.sum(fft_magnitude[mid_band]**2), 6)
        features['f_high_band_power'] = round(np.sum(fft_magnitude[high_band]**2), 6)

        # Dominant Frequency
        features['f_dominant_freq'] = features['f_peak_freq']

        # Time-Frequency Percentile features
        features['f_5th_percentile'] = round(np.percentile(fft_magnitude, 5), 6)
        features['f_25th_percentile'] = round(np.percentile(fft_magnitude, 25), 6)
        features['f_75th_percentile'] = round(np.percentile(fft_magnitude, 75), 6)
        features['f_95th_percentile'] = round(np.percentile(fft_magnitude, 95), 6)

        # Wavelet-based features
        features['f_wavelet_mean'] = round(np.mean(fft_magnitude), 6)
        features['f_wavelet_median'] = round(np.median(fft_magnitude), 6)
        features['f_wavelet_rms'] = round(np.sqrt(np.mean(fft_magnitude**2)), 6)
        features['f_wavelet_std'] = round(np.std(fft_magnitude), 6)
        features['f_wavelet_variance'] = round(np.var(fft_magnitude), 6)

        return features

    # Slice the signal into smaller chunks
    n1_slices = slice_signal(n1)

    # Extract features for each slice of the signal
    signals = []
    for signal in n1_slices:
        signals.append(extract_signal_features(signal[[column]], sampling_frequency))

    # Convert the list of features into a DataFrame
    signals = pd.DataFrame(signals)
    signals['type'] = column
    # Add additional columns for load, rpm, and id
    return signals

In [5]:
df_x=make_df(df,'x')
df_y=make_df(df,'y')
df_z=make_df(df,'z')

In [6]:
df_x['target']=[1 for n in range(0,88)] + [0 for n in range(88,180)] + [2 for n in range(180, 214)] + [1 for n in range(214, 245)]
df_y['target']=[1 for n in range(0,88)] + [0 for n in range(88,180)] + [2 for n in range(180, 214)] + [1 for n in range(214, 245)]
df_z['target']=[1 for n in range(0,88)] + [0 for n in range(88,180)] + [2 for n in range(180, 214)] + [1 for n in range(214, 245)]

In [7]:
def slice_signal(signal, slice_length=500):
    slices = [signal.iloc[i:i + slice_length] for i in range(0, len(signal), slice_length)]
    return slices[:-1]

In [8]:
pd.concat([df_x,df_y,df_z],axis=0).to_csv('data.csv',index=False)